# Tariikhna — TTS Voice Comparison

Generates the SAME narration across three engines so you can listen side by side
and pick the best for your children's comic:

1. **edge-tts** — free, no key, no limits (Microsoft neural voices)
2. **ElevenLabs** — most expressive (free tier: 10k chars/month)
3. **Google Cloud TTS** — excellent quality, generous free tier

You can run whichever engines you have keys for — each is optional. The notebook
plays every clip inline so you can compare directly.

### What you need (only for the engines you want to try):
- ElevenLabs API key: https://elevenlabs.io (free signup)
- Google Cloud TTS: a service-account JSON key (more setup - see that section)

## 1. Install

In [ ]:
!pip install edge-tts elevenlabs google-cloud-texttospeech -q
import os, asyncio
from IPython.display import Audio, display, HTML
print('Installed')

## 2. The Sample Text

We'll use a real panel from your stories. Pick something with emotion so you can
judge expressiveness. Edit this to test different lines.

In [ ]:
# A few sample lines with different emotional tones - test whichever you like
SAMPLES = {
    'gentle_grief': "Soon after, Abd al-Muttalib passed away. Muhammad was only eight years old. He was very sad to lose his grandfather, who had been so kind to him.",
    'warm_joy': "Without hesitation, Zayd said, 'I will stay with you.' He had seen such goodness in this man that he could never leave him.",
    'moral_lesson': "Abu Bakr did not free the slaves for thanks or protection. He did it only to please God. In Islam, every person is equal and precious.",
}

# Pick which sample to compare
SAMPLE_KEY = 'gentle_grief'
TEXT = SAMPLES[SAMPLE_KEY]

os.makedirs('voice_compare', exist_ok=True)
print(f'Comparing on sample: "{SAMPLE_KEY}"\n')
print(TEXT)

## 3. Engine 1 — edge-tts (free, no key)

Generates a few different voices so you can compare within edge-tts too.

In [ ]:
import edge_tts

EDGE_VOICES = {
    'edge_Jenny': 'en-US-JennyNeural',     # gentle female storyteller
    'edge_Aria': 'en-US-AriaNeural',       # warm friendly female
    'edge_Ana_child': 'en-US-AnaNeural',   # actual child voice
    'edge_Davis': 'en-US-DavisNeural',     # calm male, bedtime feel
    'edge_Sonia_UK': 'en-GB-SoniaNeural',  # British storybook warmth
}

async def gen_edge():
    for label, voice in EDGE_VOICES.items():
        out = f'voice_compare/{label}.mp3'
        try:
            comm = edge_tts.Communicate(TEXT, voice, rate='-10%')
            await comm.save(out)
            print(f'  {label} OK')
        except Exception as e:
            print(f'  {label} FAILED: {str(e)[:60]}')

await gen_edge()
# If 'await' errors in your environment: asyncio.run(gen_edge())
print('\nedge-tts done')

## 4. Engine 2 — ElevenLabs (most expressive)

Free signup gives 10k characters/month. Get a key at https://elevenlabs.io
(Profile -> API Keys). Skip this cell if you don't want to try it.

In [ ]:
ELEVENLABS_KEY = 'sk_b1489671e9cecf305d17b3107877495bc8f28452ba6d9131'   # <- paste your key, or leave '' to skip

# A few good ElevenLabs voices for warm narration (these are standard preset voices)
ELEVEN_VOICES = {
    'eleven_Rachel': 'Rachel',     # calm, warm narration
    'eleven_Bella': 'Bella',       # soft, gentle
    'eleven_Charlotte': 'Charlotte',  # warm, expressive
}

if ELEVENLABS_KEY:
    try:
        from elevenlabs.client import ElevenLabs
        client = ElevenLabs(api_key=ELEVENLABS_KEY)
        # Get available voices and map names to IDs
        available = {v.name: v.voice_id for v in client.voices.get_all().voices}
        print('Available ElevenLabs voices:', list(available.keys())[:10])
        
        for label, vname in ELEVEN_VOICES.items():
            if vname not in available:
                print(f'  {label}: voice "{vname}" not in your account, skipping')
                continue
            try:
                audio = client.text_to_speech.convert(
                    voice_id=available[vname],
                    text=TEXT,
                    model_id='eleven_multilingual_v2',
                )
                out = f'voice_compare/{label}.mp3'
                with open(out, 'wb') as f:
                    for chunk in audio:
                        f.write(chunk)
                print(f'  {label} OK')
            except Exception as e:
                print(f'  {label} FAILED: {str(e)[:80]}')
    except Exception as e:
        print(f'ElevenLabs setup error: {str(e)[:100]}')
else:
    print('No ElevenLabs key - skipping (paste a key above to try it)')

## 5. Engine 3 — Google Cloud TTS

Best quality + generous free tier, but needs setup:
1. Create a Google Cloud project, enable the **Text-to-Speech API**
2. Create a **service account** and download its JSON key
3. Upload that JSON here (Colab) or set its path (local)

Skip if you don't want the extra setup.

In [ ]:
GOOGLE_KEY_PATH = ''   # <- path to your service-account JSON, or '' to skip

# Colab: uncomment to upload the JSON key
# from google.colab import files
# up = files.upload()
# GOOGLE_KEY_PATH = list(up.keys())[0]

GOOGLE_VOICES = {
    'google_Wavenet_F': ('en-US', 'en-US-Wavenet-F'),   # warm female
    'google_Neural2_F': ('en-US', 'en-US-Neural2-F'),   # natural female
    'google_News_K': ('en-US', 'en-US-News-K'),         # storyteller-ish
}

if GOOGLE_KEY_PATH and os.path.exists(GOOGLE_KEY_PATH):
    try:
        os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = GOOGLE_KEY_PATH
        from google.cloud import texttospeech
        gclient = texttospeech.TextToSpeechClient()
        synthesis_input = texttospeech.SynthesisInput(text=TEXT)
        for label, (lang, vname) in GOOGLE_VOICES.items():
            try:
                voice = texttospeech.VoiceSelectionParams(language_code=lang, name=vname)
                audio_config = texttospeech.AudioConfig(
                    audio_encoding=texttospeech.AudioEncoding.MP3,
                    speaking_rate=0.9,   # slightly slower for kids
                )
                resp = gclient.synthesize_speech(input=synthesis_input, voice=voice, audio_config=audio_config)
                out = f'voice_compare/{label}.mp3'
                with open(out, 'wb') as f:
                    f.write(resp.audio_content)
                print(f'  {label} OK')
            except Exception as e:
                print(f'  {label} FAILED: {str(e)[:80]}')
    except Exception as e:
        print(f'Google setup error: {str(e)[:100]}')
else:
    print('No Google key - skipping (set GOOGLE_KEY_PATH above to try it)')

## 6. Listen & Compare

Every generated clip, grouped by engine, played inline.

In [ ]:
import glob

clips = sorted(glob.glob('voice_compare/*.mp3'))
print(f'Sample text: "{TEXT}"\n')
print(f'{len(clips)} clips generated:\n')

for engine in ['edge', 'eleven', 'google']:
    eng_clips = [c for c in clips if os.path.basename(c).startswith(engine)]
    if not eng_clips:
        continue
    print(f'\n===== {engine.upper()} =====')
    for c in eng_clips:
        print(os.path.basename(c).replace('.mp3',''))
        display(Audio(c))

## 7. Notes for Picking

Listen for:
- **Warmth** — does it sound like someone reading to a child, or a news anchor?
- **Emotion** — does the grief line sound gentle? Does joy sound happy?
- **Pacing** — natural pauses, not rushed
- **Pronunciation** — does it handle the names reasonably? (all engines may
  struggle with Arabic names like 'Abd al-Muttalib' - you can add phonetic
  spellings in the text if needed, e.g. 'Abdul-Muttalib')

Once you pick a winner, I can build the full generation notebook using that engine
(reading all your story JSONs, like the edge-tts one but with your chosen voice).

**Cost reminder:** your 4 stories total only a few thousand characters, so even
ElevenLabs' free 10k/month tier covers them. Google's free tier is far larger.
edge-tts is unlimited and free.